# 002 · Raising and Custom Exceptions

> **Syntax › Error Handling** · Python 3.14

Python's `raise` is C#'s `throw`. This notebook covers signalling errors yourself: raising built-in and custom exceptions, re-raising, chaining exceptions (Python's take on `InnerException`), attaching notes, and the related `assert` statement and `warnings` module.

**In this notebook you will learn:**

- `raise` vs `throw`, and re-raising with a bare `raise` (C#'s `throw;`)
- defining custom exception classes with extra attributes
- exception chaining with `raise ... from e`, `from None`, and `__cause__` / `__context__`
- adding context with `add_note()` (3.11+)
- `assert` vs `Debug.Assert`
- issuing warnings with the `warnings` module

## C# mental model

| C# | Python |
|---|---|
| `throw new ArgumentException("msg");` | `raise ValueError("msg")` |
| `throw;` (re-throw, keep stack trace) | `raise` (bare) |
| `class MyException : Exception` | `class MyError(Exception):` |
| `new MyException("msg", inner)` | `raise MyError("msg") from inner` |
| `ex.InnerException` | `e.__cause__` (explicit) / `e.__context__` (implicit) |
| `ex.Data["key"] = ...` | `e.add_note("...")` or custom attributes |
| `Debug.Assert(cond, "msg")` | `assert cond, "msg"` |
| `[Obsolete("msg")]` | `warnings.warn("msg", DeprecationWarning)` / `@warnings.deprecated` |

## `raise` — Python's `throw`

```csharp
// C#
void SetAge(int age)
{
    if (age < 0)
        throw new ArgumentOutOfRangeException(nameof(age), "age must be non-negative");
}
```

Note there is no `new` keyword in Python — you just call the class:

In [ ]:
def set_age(age: int) -> None:
    if age < 0:
        raise ValueError(f"age must be non-negative, got {age}")
    print("age set to", age)

set_age(30)
try:
    set_age(-1)
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

You can also raise the **class** itself; Python instantiates it with no arguments. And anything you raise must derive from `BaseException`:

In [ ]:
try:
    raise NotImplementedError         # same as raise NotImplementedError()
except NotImplementedError as e:
    print(f"{type(e).__name__}, message={str(e)!r}")   # instantiated with no message

try:
    raise "something went wrong"      # strings are not exceptions
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

## Re-raising with a bare `raise`

Inside an `except` block, a bare `raise` re-raises the current exception with its original traceback — exactly like C#'s `throw;`. Use it to log or clean up and then let the error continue.

```csharp
// C#
catch (Exception ex)
{
    logger.LogError(ex, "failed");
    throw;          // not "throw ex;" — that resets the stack trace
}
```

In [ ]:
def load_user(user_id: int) -> dict:
    users = {1: {"name": "alice"}}
    try:
        return users[user_id]
    except KeyError:
        print(f"  log: user {user_id} not found")
        raise                         # propagate the original KeyError

try:
    load_user(42)
except KeyError as e:
    print(f"caller got {type(e).__name__}: {e}")

Unlike C#'s `throw ex;`, `raise e` in Python does **not** reset the traceback — it keeps the original frames and adds the re-raise line. Still, bare `raise` is the idiom: it's shorter, and it's clear that you're re-raising unchanged.

In [ ]:
import traceback

def fail():
    raise ValueError("original")

def reraise_bare():
    try:
        fail()
    except ValueError:
        raise

def reraise_named():
    try:
        fail()
    except ValueError as e:
        raise e

for func in (reraise_bare, reraise_named):
    try:
        func()
    except ValueError as e:
        frames = [frame.name for frame in traceback.extract_tb(e.__traceback__)]
        print(f"{func.__name__:14} frames: {frames}")   # 'fail' is present in both

## Custom exception classes

Derive from `Exception` (never directly from `BaseException`). By convention names end in **`Error`** rather than C#'s `Exception` suffix. The simplest custom exception needs no body at all:

```csharp
// C#
public class ValidationException : Exception
{
    public ValidationException(string message) : base(message) { }
}
```

In [ ]:
class ValidationError(Exception):
    """Raised when input data is invalid."""

try:
    raise ValidationError("email is required")
except ValidationError as e:
    print(f"{type(e).__name__}: {e}")
    print(isinstance(e, Exception))

### Adding extra attributes

Give the exception structured data so handlers don't have to parse the message. Call `super().__init__(message)` so `str(e)` and `e.args` work as expected.

```csharp
// C#
public class InsufficientFundsException(decimal balance, decimal amount)
    : Exception($"cannot withdraw {amount}, balance is {balance}")
{
    public decimal Balance { get; } = balance;
    public decimal Amount { get; } = amount;
    public decimal Shortfall => Amount - Balance;
}
```

In [ ]:
class InsufficientFundsError(Exception):
    def __init__(self, balance: float, amount: float) -> None:
        super().__init__(f"cannot withdraw {amount:.2f}, balance is {balance:.2f}")
        self.balance = balance
        self.amount = amount

    @property
    def shortfall(self) -> float:
        return self.amount - self.balance

def withdraw(balance: float, amount: float) -> float:
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    return balance - amount

try:
    withdraw(50.0, 80.0)
except InsufficientFundsError as e:
    print(e)
    print("shortfall:", e.shortfall)

### An exception hierarchy for your library

A common pattern is one base exception per package, with specific subclasses. Callers can catch everything from your library with one clause, or handle specific cases.

In [ ]:
class ShopError(Exception):
    """Base class for all errors raised by the shop module."""

class OutOfStockError(ShopError):
    def __init__(self, product: str) -> None:
        super().__init__(f"{product!r} is out of stock")
        self.product = product

class PaymentDeclinedError(ShopError):
    pass

for error in (OutOfStockError("widget"), PaymentDeclinedError("card declined")):
    try:
        raise error
    except OutOfStockError as e:
        print("specific handler:", e.product)
    except ShopError as e:
        print("generic handler: ", e)

**Tip:** before inventing a new exception, check if a built-in fits — `ValueError`, `TypeError`, `KeyError`, `LookupError`, `NotImplementedError` *(see Error Handling › 001 · try / except / finally)*. You can also subclass them: `class InvalidSkuError(ValueError)` is still caught by `except ValueError`.

## Exception chaining: `raise ... from e`

When you translate a low-level exception into a higher-level one, chain them with `from` so the original isn't lost. This is Python's `InnerException`: the original is stored in `__cause__`, and the traceback shows both, joined by *"The above exception was the direct cause of the following exception"*.

```csharp
// C#
catch (FormatException ex)
{
    throw new ConfigException("invalid port", ex);   // ex becomes InnerException
}
```

In [ ]:
class ConfigError(Exception):
    pass

def parse_port(raw: str) -> int:
    try:
        return int(raw)
    except ValueError as e:
        raise ConfigError(f"invalid port {raw!r}") from e

try:
    parse_port("eighty")
except ConfigError as e:
    print(f"{type(e).__name__}: {e}")
    print("__cause__:", repr(e.__cause__))       # C#: ex.InnerException

`traceback.format_exception` shows the whole chain, the way an uncaught exception would be printed:

In [ ]:
try:
    parse_port("eighty")
except ConfigError as e:
    print("".join(traceback.format_exception(e)))

### Implicit chaining: `__context__`

If an exception is raised **while handling another** (and you don't use `from`), Python links them automatically via `__context__`. The traceback then says *"During handling of the above exception, another exception occurred"* — usually a sign of a bug in the handler, which is why explicit `from` is better when the translation is intentional.

In [ ]:
try:
    try:
        1 / 0
    except ZeroDivisionError:
        undefined_name                # a bug inside the handler
except NameError as e:
    print("__cause__:  ", repr(e.__cause__))     # None — not explicit
    print("__context__:", repr(e.__context__))   # the ZeroDivisionError

### Suppressing the chain: `from None`

Use `raise ... from None` when the original exception is an implementation detail that would only confuse the caller. It sets `__suppress_context__`, so the traceback shows only your exception.

In [ ]:
def get_setting(settings: dict, key: str) -> str:
    try:
        return settings[key]
    except KeyError:
        raise ConfigError(f"missing required setting {key!r}") from None

try:
    get_setting({}, "db_url")
except ConfigError as e:
    print("".join(traceback.format_exception(e)))
    print("__suppress_context__:", e.__suppress_context__)

| Form | `__cause__` | `__context__` | Traceback shows |
|---|---|---|---|
| `raise X` inside `except` | `None` | original | both: "During handling of…" |
| `raise X from e` | `e` | original | both: "direct cause of…" |
| `raise X from None` | `None` | original (hidden) | only `X` |

## Adding notes: `add_note()` (3.11+)

Sometimes you want to add context to an exception **without** changing its type. `add_note()` appends a string to the exception's `__notes__` list, and notes are printed below the message in the traceback. (C# would use `ex.Data` or wrap the exception.)

In [ ]:
def process_rows(rows: list[str]) -> list[int]:
    result = []
    for line_no, row in enumerate(rows, start=1):
        try:
            result.append(int(row))
        except ValueError as e:
            e.add_note(f"while processing line {line_no}")
            raise

try:
    process_rows(["1", "2", "three"])
except ValueError as e:
    print("notes:", e.__notes__)
    print("".join(traceback.format_exception(e)))

## `assert` vs `Debug.Assert`

`assert condition, message` raises `AssertionError` if the condition is false. Like C#'s `Debug.Assert` (which is compiled out of Release builds), Python's asserts are **removed** when running with `python -O`.

```csharp
// C#
Debug.Assert(items.Count > 0, "items must not be empty");
```

In [ ]:
def average(values: list[float]) -> float:
    assert values, "values must not be empty"    # internal sanity check
    return sum(values) / len(values)

print(average([1, 2, 3]))
try:
    average([])
except AssertionError as e:
    print(f"{type(e).__name__}: {e}")

⚠️ **Gotcha:** never use `assert` to validate user input or arguments of a public API — with `-O` the check silently disappears. Raise `ValueError`/`TypeError` instead. Also beware of the parentheses trap: `assert (cond, "msg")` asserts a non-empty **tuple**, which is always true.

In [ ]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    code = compile('assert (1 == 2, "never fails")', "<demo>", "exec")
    exec(code)                        # no AssertionError!

print("assert passed?!  compiler said:", caught[0].message)

## The `warnings` module (briefly)

A *warning* tells the caller about something questionable — a deprecated function, a suspicious argument — **without** stopping the program. It's printed once per location by default and can be filtered, or turned into an error (`python -W error`).

C#'s closest equivalent is the compile-time `[Obsolete]` attribute; Python's warnings are issued at **runtime**.

```csharp
// C#
[Obsolete("Use CalculateTotal instead")]
public decimal Total() => CalculateTotal();
```

In [ ]:
import warnings

def old_total(prices: list[float]) -> float:
    warnings.warn("old_total() is deprecated; use sum() instead",
                  DeprecationWarning, stacklevel=2)   # stacklevel=2 points at the caller
    return sum(prices)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print(old_total([1.0, 2.5]))

for w in caught:
    print(f"{w.category.__name__}: {w.message}")

Since 3.13, the `@warnings.deprecated` decorator (PEP 702) is the direct counterpart of `[Obsolete]`: it warns at runtime **and** type checkers flag the call statically.

In [ ]:
@warnings.deprecated("Use new_api() instead")
def legacy_api() -> str:
    return "legacy result"

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    print(legacy_api())

print(f"{caught[0].category.__name__}: {caught[0].message}")

Warnings can also be escalated to exceptions — handy in test suites:

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("error")    # turn every warning into an exception
    try:
        old_total([1.0])
    except DeprecationWarning as e:
        print(f"{type(e).__name__}: {e}")

## Summary

| Task | C# | Python |
|---|---|---|
| Throw | `throw new ArgumentException("m")` | `raise ValueError("m")` |
| Re-throw | `throw;` | `raise` |
| Custom exception | `class FooException : Exception` | `class FooError(Exception):` |
| Extra data | properties on the exception | attributes set in `__init__` |
| Wrap with inner | `new X("m", inner)` | `raise X("m") from inner` |
| Inner exception | `ex.InnerException` | `e.__cause__` / `e.__context__` |
| Hide the inner | — | `raise X("m") from None` |
| Add context | `ex.Data[...]` | `e.add_note("...")` |
| Debug check | `Debug.Assert(c, "m")` | `assert c, "m"` |
| Deprecation | `[Obsolete]` | `@warnings.deprecated` / `warnings.warn(..., DeprecationWarning)` |

## Exercises

1. Create a `NegativeQuantityError(ValueError)` with a `quantity` attribute, and a function `add_to_cart(qty)` that raises it for `qty < 0`. Show that `except ValueError` catches it.
2. Write `load_json_config(text)` that calls `json.loads(text)` and translates `json.JSONDecodeError` into a `ConfigError`, keeping the original as `__cause__`.
3. Write `parse_all(values)` that converts strings to `float`, and when one fails, adds a note naming the offending index before re-raising.
4. Why is `assert isinstance(amount, int)` a poor way to validate an argument to a public function?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
# 1
class NegativeQuantityError(ValueError):
    def __init__(self, quantity: int) -> None:
        super().__init__(f"quantity must be >= 0, got {quantity}")
        self.quantity = quantity

def add_to_cart(qty: int) -> None:
    if qty < 0:
        raise NegativeQuantityError(qty)

try:
    add_to_cart(-2)
except ValueError as e:
    print(e, e.quantity)

# 2
import json

class ConfigError(Exception):
    pass

def load_json_config(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ConfigError("config is not valid JSON") from e

# 3
def parse_all(values: list[str]) -> list[float]:
    result = []
    for i, v in enumerate(values):
        try:
            result.append(float(v))
        except ValueError as e:
            e.add_note(f"at index {i}")
            raise
    return result

# 4 — asserts are stripped with `python -O`, so the check may not run at all,
#     and AssertionError is the wrong type for bad arguments; raise TypeError.
```

</details>